# 01 — Fields, FFTs, and power spectra

For a periodic cube of volume $V=L^3$ and cell width $\Delta x$,

$$
\delta_{\mathbf k}=(\Delta x)^3\,\mathrm{FFT}[\delta],
\qquad
\widehat P(\mathbf k)=\frac{|\delta_{\mathbf k}|^2}{V}.
$$

We use the full FFT grid here because it makes the $+\mathbf k$ and $-\mathbf k$ mode count explicit.

In [ ]:
from pathlib import Path
import sys, os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

ROOT = Path(os.path.dirname(os.path.abspath('.'))).parent

OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 2601
COLORS = {
    "blue": "#2D6A9F",
    "orange": "#E6862E",
    "green": "#3A8D72",
    "purple": "#7656A5",
    "gray": "#626C78",
}

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
})

def savefig(fig, name):
    fig.text(0.995, 0.005, "analytic-fixture", ha="right", va="bottom",
             fontsize=7, color=COLORS["gray"])
    fig.savefig(OUTPUT_DIR / name, bbox_inches="tight")

In [ ]:
n = 64
box_size = 500.0  # h^-1 Mpc
volume = box_size**3
dx = box_size / n
k_fundamental = 2 * np.pi / box_size
k_nyquist = np.pi / dx
edges = np.arange(0.5 * k_fundamental, k_nyquist + 2.5 * k_fundamental,
                  2 * k_fundamental)

In [ ]:
def k_magnitude_3d(n, box_size):
    dx = box_size / n
    k_1d = 2 * np.pi * np.fft.fftfreq(n, d=dx)
    kx, ky, kz = np.meshgrid(k_1d, k_1d, k_1d, indexing="ij")
    k_magnitude = np.sqrt(kx**2 + ky**2 + kz**2)
    return k_magnitude


def power_spectrum(field_a, box_size, edges, field_b=None):
    # Use field_a twice for an auto-spectrum.
    if field_b is None:
        field_b = field_a

    n = field_a.shape[0]
    dx = box_size / n
    volume = box_size**3

    # Convert NumPy's FFT sum to our physical Fourier convention.
    field_a_k = dx**3 * np.fft.fftn(field_a)
    field_b_k = dx**3 * np.fft.fftn(field_b)

    # This is the power of every Fourier-grid mode.
    power_3d = np.real(field_a_k * np.conj(field_b_k)) / volume
    k_modes = k_magnitude_3d(n, box_size)

    mean_k = []
    mean_power = []
    number_of_modes = []

    # Average all modes lying in each spherical k-shell.
    for i in range(len(edges) - 1):
        in_shell = (k_modes >= edges[i]) & (k_modes < edges[i + 1])
        count = np.sum(in_shell)
        number_of_modes.append(count)

        if count > 0:
            mean_k.append(np.mean(k_modes[in_shell]))
            mean_power.append(np.mean(power_3d[in_shell]))
        else:
            mean_k.append(np.nan)
            mean_power.append(np.nan)

    return np.array(mean_k), np.array(mean_power), np.array(number_of_modes)

## A sinusoid fixes the convention

For $\delta=A\cos(2\pi m x/L)$, the two Fourier peaks lie at $k_x=\pm2\pi m/L$ and each has power $A^2V/4$.

In [ ]:
amplitude, mode_number = 0.8, 4
x = np.arange(n) * dx
sine = amplitude * np.cos(2 * np.pi * mode_number * x[:, None, None] / box_size)
sine = np.broadcast_to(sine, (n, n, n)).copy()

sine_k = dx**3 * np.fft.fftn(sine)
mode_power = np.abs(sine_k)**2 / volume
expected_k = 2 * np.pi * mode_number / box_size
expected_power = amplitude**2 * volume / 4
fourier_variance = mode_power.sum() / volume

print(f"peak k: {expected_k:.5f} h Mpc^-1")
print(f"mode power: {mode_power[mode_number, 0, 0]:.4e}")
print(f"real/Fourier variance: {np.mean(sine**2):.6f} / {fourier_variance:.6f}")
assert np.isclose(mode_power[mode_number, 0, 0], expected_power)
assert np.isclose(np.mean(sine**2), fourier_variance)

k, p_sine, counts = power_spectrum(sine, box_size, edges)
plane = np.log10(np.fft.fftshift(mode_power[:, :, 0]) + expected_power * 1e-14)
k_axis = np.fft.fftshift(2 * np.pi * np.fft.fftfreq(n, d=dx))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
image = axes[0].imshow(
    plane.T, origin="lower", cmap="magma",
    extent=[k_axis[0], k_axis[-1], k_axis[0], k_axis[-1]],
)
axes[0].set(title="The two Fourier peaks", xlabel=r"$k_x$",
            ylabel=r"$k_y$")
axes[0].grid(False)
fig.colorbar(image, ax=axes[0], label=r"$\log_{10}$ mode power")
axes[1].loglog(k, np.where(p_sine > 0, p_sine, np.nan), "o-",
               color=COLORS["blue"])
axes[1].axvline(expected_k, color=COLORS["orange"], ls="--")
axes[1].set(title="Isotropic shell average", xlabel=r"$k\;[h\,\mathrm{Mpc}^{-1}]$",
            ylabel=r"$P(k)\;[(h^{-1}\,\mathrm{Mpc})^3]$")
savefig(fig, "01_sinusoid_test.png")
plt.show()

## White noise and a shaped Gaussian field

Grid white noise with variance $\sigma^2$ has $\langle P\rangle=\sigma^2(\Delta x)^3$. Filtering its Fourier modes by $\sqrt{P_{\rm target}(k)/(\Delta x)^3}$ produces a Gaussian field with the chosen spectral shape.

In [ ]:
def target_power(k):
    q = np.maximum(k, 1e-12) / 0.10
    return 1800.0 * q / (1.0 + q**4)


def gaussian_field(seed):
    rng = np.random.default_rng(seed)
    white = rng.normal(size=(n, n, n))
    k_modes = k_magnitude_3d(n, box_size)
    filt = np.sqrt(target_power(k_modes) / dx**3)
    filt[0, 0, 0] = 0.0
    return np.fft.ifftn(np.fft.fftn(white) * filt).real


rng = np.random.default_rng(SEED)
sigma = 0.7
white = rng.normal(scale=sigma, size=(n, n, n))
field = gaussian_field(SEED + 1)
k, p_white, counts = power_spectrum(white, box_size, edges)
_, p_field, _ = power_spectrum(field, box_size, edges)
expected_white_power = sigma**2 * dx**3

well_sampled = (counts > 100) & (k < 0.8 * k_nyquist)
white_ratio = np.nanmedian(p_white[well_sampled] / expected_white_power)
print(f"median measured/expected white-noise power: {white_ratio:.3f}")
assert abs(white_ratio - 1) < 0.15

fig, axes = plt.subplots(1, 3, figsize=(13, 3.7), constrained_layout=True)
image = axes[0].imshow(field[:, :, n // 2].T, origin="lower", cmap="RdBu_r",
                       extent=[0, box_size, 0, box_size])
axes[0].set(title="Toy Gaussian field", xlabel=r"$x\;[h^{-1}\,\mathrm{Mpc}]$",
            ylabel=r"$y\;[h^{-1}\,\mathrm{Mpc}]$")
axes[0].grid(False)
fig.colorbar(image, ax=axes[0], label=r"$\delta$")
axes[1].loglog(k, p_white, "o", ms=4, color=COLORS["purple"])
axes[1].axhline(expected_white_power, color=COLORS["gray"], ls="--")
axes[1].set(title="White-noise normalization", xlabel=r"$k$",
            ylabel=r"$P(k)$")
axes[2].loglog(k, p_field, "o", ms=4, color=COLORS["blue"], label="measured")
axes[2].loglog(k, target_power(k), color=COLORS["orange"], label="target")
axes[2].set(title="Recovering a target spectrum", xlabel=r"$k$",
            ylabel=r"$P(k)$")
axes[2].legend()
savefig(fig, "01_power_validation.png")
plt.show()

## Mode counts, windows, and cross-correlation

This cell contains three small experiments:

1. Compare box-to-box scatter with $\sqrt{2/N_{\rm modes}}$.
2. Apply the separable CIC-like window and try a regularized deconvolution.
3. Measure $r(k)=P_{12}(k)/\sqrt{P_{11}(k)P_{22}(k)}$.

In [ ]:
# 1. Finite-volume scatter
number_of_boxes = 12
power_from_each_box = []

for box_number in range(number_of_boxes):
    box_seed = SEED + 10 + box_number
    new_field = gaussian_field(box_seed)
    _, new_power, _ = power_spectrum(new_field, box_size, edges)
    power_from_each_box.append(new_power)

power_from_each_box = np.array(power_from_each_box)
average_power = np.nanmean(power_from_each_box, axis=0)
scatter_in_power = np.nanstd(power_from_each_box, axis=0, ddof=1)
fractional_scatter = scatter_in_power / average_power

gaussian_prediction = np.full(len(k), np.nan)
bins_with_modes = counts > 0
gaussian_prediction[bins_with_modes] = np.sqrt(
    2.0 / counts[bins_with_modes]
)


# 2. Grid window and regularized deconvolution
k_1d = 2 * np.pi * np.fft.fftfreq(n, d=dx)
kx, ky, kz = np.meshgrid(k_1d, k_1d, k_1d, indexing="ij")
k_modes = np.sqrt(kx**2 + ky**2 + kz**2)

window_x = np.sinc(kx * dx / (2 * np.pi))**2
window_y = np.sinc(ky * dx / (2 * np.pi))**2
window_z = np.sinc(kz * dx / (2 * np.pi))**2
cic_window = window_x * window_y * window_z

field_k = np.fft.fftn(field)
windowed_field = np.fft.ifftn(field_k * cic_window).real

noise_rng = np.random.default_rng(SEED + 30)
measurement_noise = 0.1 * field.std() * noise_rng.normal(size=field.shape)
observed_field = windowed_field + measurement_noise

window_floor = 0.2
regularized_window = np.maximum(cic_window, window_floor)
deconvolved_field = np.fft.ifftn(
    np.fft.fftn(observed_field) / regularized_window
).real

_, observed_power, _ = power_spectrum(observed_field, box_size, edges)
_, deconvolved_power, _ = power_spectrum(deconvolved_field, box_size, edges)


# 3. Cross-correlation between two fields
independent_field = gaussian_field(SEED + 40)
signal_transfer = np.exp(-0.5 * (k_modes / (0.7 * k_nyquist))**4)
independent_weight = 0.75 * (k_modes / k_nyquist)**2

comparison_field_k = (
    signal_transfer * np.fft.fftn(field)
    + independent_weight * np.fft.fftn(independent_field)
)
comparison_field = np.fft.ifftn(comparison_field_k).real

_, power_11, _ = power_spectrum(field, box_size, edges)
_, power_22, _ = power_spectrum(comparison_field, box_size, edges)
_, power_12, _ = power_spectrum(field, box_size, edges, comparison_field)
correlation = power_12 / np.sqrt(power_11 * power_22)



# Plot the three experiments.
scatter_bins = bins_with_modes & np.isfinite(fractional_scatter)
power_bins = bins_with_modes & (p_field > 0) & (k < 0.98 * k_nyquist)
correlation_bins = power_bins & (power_11 > 0) & (power_22 > 0)

fig, (ax_scatter, ax_window, ax_cross) = plt.subplots(
    1, 3, figsize=(13.5, 3.7), constrained_layout=True
)

ax_scatter.loglog(
    k[scatter_bins], fractional_scatter[scatter_bins], "o",
    color=COLORS["purple"], label=f"{number_of_boxes} boxes",
)
ax_scatter.loglog(
    k[scatter_bins], gaussian_prediction[scatter_bins],
    color=COLORS["gray"], label=r"$\sqrt{2/N_{\rm modes}}$",
)
ax_scatter.set(
    title="Finite-volume scatter",
    xlabel=r"$k$",
    ylabel=r"$\sigma_P/\langle P\rangle$",
)
ax_scatter.legend()

ax_window.semilogx(
    k[power_bins], observed_power[power_bins] / p_field[power_bins], "o-",
    color=COLORS["orange"], label="windowed + noise",
)
ax_window.semilogx(
    k[power_bins], deconvolved_power[power_bins] / p_field[power_bins], "o-",
    color=COLORS["green"], label="regularized deconvolution",
)
ax_window.axhline(1, color=COLORS["gray"])
ax_window.set(
    title="Window and deconvolution",
    xlabel=r"$k$",
    ylabel=r"$P/P_{\rm original}$",
    ylim=(0.45, 1.75),
)
ax_window.legend(fontsize=8)

ax_cross.semilogx(
    k[correlation_bins], correlation[correlation_bins], "o-",
    color=COLORS["blue"],
)
ax_cross.axhline(1, color=COLORS["gray"])
ax_cross.set(
    title="Mode-by-mode fidelity",
    xlabel=r"$k$",
    ylabel=r"$r(k)$",
    ylim=(0, 1.05),
)

savefig(fig, "01_variance_window_cross.png")
plt.show()

The estimator is now fixed by two transparent limits: a sinusoid and white noise. Mode counts explain the large-scale scatter; $r(k)$ tests information that two auto-spectra alone cannot.